# Text Sentiment Classification with Optimized LSTM
## HW4 - GPU Accelerated Training on Colab

### Setup Instructions:
1. **Enable GPU**: Go to `Runtime` → `Change runtime type` → Select `GPU`
2. **Upload Data**: Upload `train.csv`, `train_unlabel.csv`, `test.csv` to the same directory
3. **Run All Cells**: Execute all cells in order

### Key Optimizations:
- **Model**: 2-layer Bidirectional LSTM (256 hidden units, 300 embedding dim)
- **Training**: AdamW + CosineAnnealing scheduler + Early stopping
- **Self-training**: 3 iterations with unlabeled data
- **Regularization**: Dropout (0.4) + BatchNorm + Gradient clipping

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Import required libraries
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import re
import os
from collections import Counter
import random
from torch.utils import data

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
pattern = r'\w+|[^\w\s]'
path_prefix = './'

In [ ]:
# ============== Data Loading Functions ==============
def load_training_data(path='train.csv'):
    data = pd.read_csv(path).to_numpy()
    lines = data[:,0]
    x = [re.findall(pattern, line) for line in lines]
    if 'unlabel' not in path:
        y = data[:,1]
        return x, y
    else:
        return x

def load_testing_data(path='test.csv'):
    data = pd.read_csv(path).to_numpy()
    lines = data[:,1]
    x = [re.findall(pattern, line) for line in lines]
    return x

def evaluation(outputs, labels):
    outputs[outputs>=0.5] = 1
    outputs[outputs<0.5] = 0
    correct = torch.sum(torch.eq(outputs, labels)).item()
    return correct

In [ ]:
# ============== Simple Vocabulary-Based Embedding ==============
class SimpleVocabEmbedding:
    """
    A simple vocabulary-based embedding approach without external packages.
    Uses randomized embeddings initialized from a normal distribution.
    """
    def __init__(self, sentences, vector_size=300, min_count=1):
        self.vector_size = vector_size
        self.min_count = min_count
        self.wv = {}
        self._build_vocab(sentences)
    
    def _count_words(self, sentences):
        word_counts = Counter()
        for sentence in sentences:
            word_counts.update(sentence)
        return word_counts
    
    def _build_vocab(self, sentences):
        print("Building vocabulary...")
        word_counts = self._count_words(sentences)
        vocab = [word for word, count in word_counts.items() if count >= self.min_count]
        print(f"Vocabulary size: {len(vocab)} (min_count={self.min_count})")
        for word in vocab:
            self.wv[word] = np.random.randn(self.vector_size).astype(np.float32) * np.sqrt(2.0 / self.vector_size)
        print("Embeddings initialized.")
    
    def save(self, path):
        torch.save(self.wv, path)
    
    @classmethod
    def load(cls, path):
        model = cls.__new__(cls)
        model.wv = torch.load(path, weights_only=False)
        return model

In [ ]:
# ============== Preprocessing ==============
class Preprocess():
    def __init__(self, sentences, sen_len, w2v_path="./w2v.model"):
        self.w2v_path = w2v_path
        self.sentences = sentences
        self.sen_len = sen_len
        self.idx2word = []
        self.word2idx = {}
        self.embedding_matrix = []
    
    def get_w2v_model(self):
        self.wv_dict = torch.load(self.w2v_path, weights_only=False)
        self.embedding_dim = len(list(self.wv_dict.values())[0])
    
    def add_embedding(self, word):
        vector = torch.empty(1, self.embedding_dim)
        torch.nn.init.uniform_(vector)
        self.word2idx[word] = len(self.word2idx)
        self.idx2word.append(word)
        self.embedding_matrix = torch.cat([self.embedding_matrix, vector], 0)
    
    def make_embedding(self, load=True):
        print("Get embedding ...")
        if load:
            print("loading word to vec model ...")
            self.get_w2v_model()
        else:
            raise NotImplementedError
        for i, word in enumerate(self.wv_dict):
            print(f'get words #{i+1}', end='\r')
            self.word2idx[word] = len(self.word2idx)
            self.idx2word.append(word)
            self.embedding_matrix.append(torch.tensor(self.wv_dict[word], dtype=torch.float32))
        print('')
        self.embedding_matrix = torch.stack(self.embedding_matrix)
        self.add_embedding("<PAD>")
        self.add_embedding("<UNK>")
        print(f"total words: {len(self.embedding_matrix)}")
        return self.embedding_matrix
    
    def pad_sequence(self, sentence):
        if len(sentence) > self.sen_len:
            sentence = sentence[:self.sen_len]
        else:
            pad_len = self.sen_len - len(sentence)
            for _ in range(pad_len):
                sentence.append(self.word2idx["<PAD>"])
        assert len(sentence) == self.sen_len
        return sentence
    
    def sentence_word2idx(self):
        sentence_list = []
        for i, sen in enumerate(self.sentences):
            print(f'sentence count #{i+1}', end='\r')
            sentence_idx = []
            for word in sen:
                if word in self.word2idx:
                    sentence_idx.append(self.word2idx[word])
                else:
                    sentence_idx.append(self.word2idx["<UNK>"])
            sentence_idx = self.pad_sequence(sentence_idx)
            sentence_list.append(sentence_idx)
        return torch.LongTensor(sentence_list)
    
    def labels_to_tensor(self, y):
        y = [int(label) for label in y]
        return torch.LongTensor(y)

In [ ]:
# ============== Dataset ==============
class SenDataset(data.Dataset):
    def __init__(self, X, y):
        self.data = X
        self.label = y
    def __getitem__(self, idx):
        if self.label is None: 
            return self.data[idx]
        return self.data[idx], self.label[idx]
    def __len__(self):
        return len(self.data)

In [ ]:
# ============== Improved LSTM Model ==============
class LSTM_Net(nn.Module):
    def __init__(self, embedding, embedding_dim, hidden_dim, num_layers, dropout=0.5, fix_embedding=True, bidirectional=False):
        super(LSTM_Net, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(embedding.size(0), embedding.size(1))
        self.embedding.weight = nn.Parameter(embedding)
        self.embedding.weight.requires_grad = False if fix_embedding else True
        self.embedding_dim = embedding.size(1)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        # Enhanced classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_output_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    
    def forward(self, inputs):
        embedded = self.embedding(inputs)
        lstm_out, (hidden, cell) = self.lstm(embedded, None)
        
        if self.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]
        
        output = self.classifier(hidden)
        return output

In [ ]:
# ============== Training Function ==============
def training(batch_size, n_epoch, lr, model_dir, train, valid, model, device):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\nstart training, parameter total:{total}, trainable:{trainable}\n')
    
    model.train()
    criterion = nn.BCELoss()
    t_batch = len(train)
    v_batch = len(valid)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=3, T_mult=2, eta_min=1e-6)
    
    total_loss, total_acc, best_acc = 0, 0, 0
    patience_counter = 0
    max_patience = 5
    
    for epoch in range(n_epoch):
        model.train()
        total_loss, total_acc = 0, 0
        
        for i, (inputs, labels) in enumerate(train):
            inputs = inputs.to(device, dtype=torch.long)
            labels = labels.to(device, dtype=torch.float)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            outputs = outputs.squeeze()
            
            loss = criterion(outputs, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            correct = evaluation(outputs.detach(), labels)
            total_acc += correct
            total_loss += loss.item()
            
            if (i + 1) % 50 == 0:
                print(f'[ Epoch{epoch+1}: {i+1}/{t_batch} ] loss:{loss.item():.3f} acc:{correct*100/batch_size:.3f} ', end='\r')
        
        train_acc = total_acc / (t_batch * batch_size)
        print(f'\nTrain | Loss:{total_loss/t_batch:.5f} Acc: {train_acc*100:.3f}')
        
        model.eval()
        with torch.no_grad():
            val_loss, val_acc = 0, 0
            for i, (inputs, labels) in enumerate(valid):
                inputs = inputs.to(device, dtype=torch.long)
                labels = labels.to(device, dtype=torch.float)
                
                outputs = model(inputs)
                outputs = outputs.squeeze()
                
                loss = criterion(outputs, labels)
                correct = evaluation(outputs, labels)
                val_acc += correct
                val_loss += loss.item()
            
            val_acc = val_acc / (v_batch * batch_size)
            print(f"Valid | Loss:{val_loss/v_batch:.5f} Acc: {val_acc*100:.3f} ")
            
            scheduler.step()
            
            if val_acc > best_acc:
                best_acc = val_acc
                patience_counter = 0
                torch.save(model, f"{model_dir}/ckpt.model")
                print(f'saving model with acc {val_acc*100:.3f}')
            else:
                patience_counter += 1
                if patience_counter >= max_patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break
        
        print('-----------------------------------------------')
    
    return best_acc

In [ ]:
# ============== Testing Function ==============
def testing(batch_size, test_loader, model, device):
    model.eval()
    ret_output = []
    with torch.no_grad():
        for i, inputs in enumerate(test_loader):
            inputs = inputs.to(device, dtype=torch.long)
            outputs = model(inputs)
            outputs = outputs.squeeze()
            outputs[outputs>=0.5] = 1
            outputs[outputs<0.5] = 0
            ret_output += outputs.int().tolist()
    
    return ret_output

# ============== Self-Training ==============
def self_training_iteration(model, unlabeled_loader, device, threshold=0.95):
    model.eval()
    pseudo_labeled_data = []
    
    with torch.no_grad():
        for inputs in unlabeled_loader:
            inputs = inputs.to(device, dtype=torch.long)
            outputs = model(inputs)
            outputs = outputs.squeeze()
            
            confident_mask = ((outputs > threshold) | (outputs < (1 - threshold)))
            
            for i in range(len(inputs)):
                if confident_mask[i]:
                    pseudo_label = int(outputs[i] >= 0.5)
                    pseudo_labeled_data.append((inputs[i].cpu(), pseudo_label))
    
    return pseudo_labeled_data

In [ ]:
# ============== Main Training Pipeline ==============
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Paths
train_with_label = os.path.join(path_prefix, 'train.csv')
train_no_label = os.path.join(path_prefix, 'train_unlabel.csv')
testing_data = os.path.join(path_prefix, 'test.csv')
w2v_path = os.path.join(path_prefix, 'w2v_all_optimized.model')

# Optimized Hyperparameters
sen_len = 50
fix_embedding = False
batch_size = 128
epoch = 15
lr = 0.002
hidden_dim = 256
num_layers = 2
dropout = 0.4
bidirectional = True
model_dir = path_prefix

In [ ]:
# Load data
print("loading training data ...")
train_x, y = load_training_data(train_with_label)
train_x_no_label = load_training_data(train_no_label)

print("loading testing data ...")
test_x = load_testing_data(testing_data)

# Build embeddings
print("\n=== Building Vocabulary Embeddings ===")
all_sentences = train_x + train_x_no_label + test_x
w2v_model = SimpleVocabEmbedding(all_sentences, vector_size=300, min_count=1)
w2v_model.save(w2v_path)
print(f"Embeddings saved to {w2v_path}\n")

In [ ]:
# Preprocess labeled data
print("\n=== Preprocessing Labeled Data ===")
preprocess = Preprocess(train_x, sen_len, w2v_path=w2v_path)
embedding = preprocess.make_embedding(load=True)
train_x_tensor = preprocess.sentence_word2idx()
y_tensor = preprocess.labels_to_tensor(y)

# Split train/val
split_idx = int(len(train_x_tensor) * 0.9)
X_train, X_val = train_x_tensor[:split_idx], train_x_tensor[split_idx:]
y_train, y_val = y_tensor[:split_idx], y_tensor[split_idx:]

print(f"Training samples: {len(X_train)}, Validation samples: {len(X_val)}")

In [ ]:
# Create datasets and loaders
train_dataset = SenDataset(X=X_train, y=y_train)
val_dataset = SenDataset(X=X_val, y=y_val)

train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

val_loader = torch.utils.data.DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Initialize model
model = LSTM_Net(
    embedding, 
    embedding_dim=300, 
    hidden_dim=hidden_dim, 
    num_layers=num_layers, 
    dropout=dropout, 
    fix_embedding=fix_embedding,
    bidirectional=bidirectional
)
model = model.to(device)

# Initial training
print("\n=== Initial Training ===")
best_acc = training(batch_size, epoch, lr, model_dir, train_loader, val_loader, model, device)

In [ ]:
# Self-training with unlabeled data
print("\n=== Self-Training with Unlabeled Data ===")
preprocess_unlabeled = Preprocess(train_x_no_label, sen_len, w2v_path=w2v_path)
_ = preprocess_unlabeled.make_embedding(load=True)
unlabeled_x = preprocess_unlabeled.sentence_word2idx()

unlabeled_dataset = SenDataset(X=unlabeled_x, y=None)
unlabeled_loader = torch.utils.data.DataLoader(
    dataset=unlabeled_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

for iteration in range(3):
    print(f"\n--- Self-training iteration {iteration + 1} ---")
    threshold = 0.95 - iteration * 0.05
    print(f"Generating pseudo-labels with threshold={threshold}...")
    pseudo_data = self_training_iteration(model, unlabeled_loader, device, threshold=threshold)
    print(f"Generated {len(pseudo_data)} pseudo-labeled samples")
    
    if len(pseudo_data) > 0:
        combined_X = torch.cat([X_train, torch.stack([x for x, _ in pseudo_data])], dim=0)
        combined_y = torch.cat([y_train, torch.tensor([y for _, y in pseudo_data])], dim=0)
        print(f"Combined training set size: {len(combined_X)}")
        
        combined_dataset = SenDataset(X=combined_X, y=combined_y)
        combined_loader = torch.utils.data.DataLoader(
            dataset=combined_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            drop_last=True
        )
        
        model = torch.load(os.path.join(model_dir, 'ckpt.model'), weights_only=False)
        model = model.to(device)
        
        print(f"\n=== Fine-tuning iteration {iteration + 1} ===")
        training(batch_size, 5, lr * (0.5 ** (iteration + 1)), model_dir, combined_loader, val_loader, model, device)

In [ ]:
# Final prediction
print("\n=== Final Prediction ===")
preprocess_test = Preprocess(test_x, sen_len, w2v_path=w2v_path)
_ = preprocess_test.make_embedding(load=True)
test_x_tensor = preprocess_test.sentence_word2idx()

test_dataset = SenDataset(X=test_x_tensor, y=None)
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print('load model ...')
model = torch.load(os.path.join(model_dir, 'ckpt.model'), weights_only=False)
outputs = testing(batch_size, test_loader, model, device)

# Save predictions
tmp = pd.DataFrame({"id":[str(i) for i in range(len(test_x))], "labels":outputs})
print("save csv ...")
tmp.to_csv(os.path.join(path_prefix, 'predict_optimized.csv'), index=False)
print("Finish Predicting!")
print(f"Best validation accuracy: {best_acc*100:.3f}%")